In [1]:
import json
import os
import pandas as pd

In [2]:
results_dir = '/data/rosa/work_in_progress/dictionary_learning_demo/._eval_on_shoe__data_rosa_work_in_progress_compositional_interpretability_outputs_shoe_simple_two_level_lr0.0005_epochs30_batch8_warmup100_pythia_cls_head_batch_top_k'
# '/data/rosa/work_in_progress/dictionary_learning_demo/SAEs_k_search_combined'

# '/data/rosa/work_in_progress/dictionary_learning_demo/._k_search__data_rosa_work_in_progress_compositional_interpretability_outputs_shoe_simple_two_level_lr0.0005_epochs30_batch8_warmup100_pythia_cls_head_batch_top_k'
# './SAEs_dev_shoe_simple_two_level_standard_lambda1e-1'
# './SAEs_dev_shoe_simple_two_level_batch_top_k_640'

In [3]:
results = {}

for submodule_dir in os.listdir(results_dir):
    for trainer_dir in os.listdir(os.path.join(results_dir, submodule_dir)):
        folder = os.path.join(submodule_dir, trainer_dir)
        run_dir = os.path.join(results_dir, folder)
        try:
            with open(os.path.join(run_dir, 'eval_results.json')) as f:
                metrics = json.load(f)
                config_file = os.path.join(run_dir, 'config.json')
                with open(config_file) as f:
                    config = json.load(f)
                    # l1_penalty = config['trainer']['l1_penalty']
                    k = config['trainer']['k']
                    # target_l0 = config['trainer']['target_l0']
                # results[f'{submodule_dir}_lambda{l1_penalty}'] = metrics
                results[f'{submodule_dir}_k{k}'] = metrics
                # results[f'{submodule_dir}_target_l0{target_l0}'] = metrics
        except FileNotFoundError:
            print(f'No eval_results.json in {run_dir}')

In [4]:
# sort by submodule name
results = dict(sorted(results.items()))

In [5]:
results

{'resid_out_5_k150': {'l2_loss': 4.609486212730408,
  'l1_loss': 513.0222514343262,
  'l0': 187.456796875,
  'frac_variance_explained': 0.9824023222923279,
  'cossim': 0.9974720680713653,
  'l2_ratio': 0.9967644336819649,
  'relative_reconstruction_bias': 0.998099265396595,
  'loss_original': 9.510973978042603,
  'loss_reconstructed': 9.562131333351136,
  'loss_zero': 8.260828738212586,
  'frac_recovered': 1.0889301386475563,
  'frac_alive': 0.05633544921875,
  'hyperparameters': {'n_inputs': 200, 'context_length': 64}},
 'resid_out_5_k300': {'l2_loss': 3.170931648015976,
  'l1_loss': 760.5723962402344,
  'l0': 333.030625,
  'frac_variance_explained': 0.9913517883419991,
  'cossim': 0.9988564962148666,
  'l2_ratio': 0.9976987478137016,
  'relative_reconstruction_bias': 0.9968207105994225,
  'loss_original': 9.510973978042603,
  'loss_reconstructed': 9.557154107093812,
  'loss_zero': 8.260828738212586,
  'frac_recovered': 1.0614450592547655,
  'frac_alive': 0.04449462890625,
  'hyperpar

In [6]:
# get a df where the columns are submodule names, frac_variance_explained, l1_loss, l0, frac_alive, and frac_recovered
df = pd.DataFrame(results).T
df = df[['frac_variance_explained', 'l1_loss', 'l0', 'frac_alive', 'frac_recovered', 'loss_original', 'loss_reconstructed']]

# add a column for the difference between the original and reconstructed loss
df['loss_diff'] = df['loss_original'] - df['loss_reconstructed']
df['loss_diff'] = df['loss_diff'].abs()
# drop the original and reconstructed loss columns
df = df.drop(columns=['loss_original', 'loss_reconstructed'])

# sort by submodule names
df = df.sort_index()

# turn fractions into percentages
df['frac_alive'] = 100*df['frac_alive']
df['frac_recovered'] = 100*df['frac_recovered']
df['frac_variance_explained'] = 100*df['frac_variance_explained']
df['l0'] = df['l0'].astype(int)
df['l1_loss'] = df['l1_loss'].astype(int)
df['frac_alive'] = df['frac_alive'].astype(int)
df['frac_recovered'] = df['frac_recovered'].astype(int)
df['frac_variance_explained'] = df['frac_variance_explained'].astype(int)
df = df.apply(pd.to_numeric, errors='coerce')
# keep the first two decimal places for loss_diff
df['loss_diff'] = df['loss_diff'].round(2)
# move the frac_recovery column and its values to the end
df = df[['frac_variance_explained', 'l1_loss', 'l0', 'frac_alive', 'loss_diff', 'frac_recovered']]

# add % sign to the values in the first column
df['frac_variance_explained'] = df['frac_variance_explained'].astype(str) + '%'
df['frac_alive'] = df['frac_alive'].astype(str) + '%'
df['frac_recovered'] = df['frac_recovered'].astype(str) + '%'

# rename the columns
df.columns = ['% Variance Explained', 'L1', 'L0', '% Alive', 'CE Diff', '% CE Recovered']

In [7]:
df

,% Variance Explained,L1,L0,% Alive,CE Diff,% CE Recovered
resid_out_5_k150,98%,513,187,5%,0.05,108%
resid_out_5_k300,99%,760,333,4%,0.05,106%
resid_out_5_k60,97%,151,78,8%,0.04,109%
